# 最小循环：模型请求怎样变成工具执行

本章学习目标：

- 调用模型，获得普通文本和结构化 JSON。
- 定义工具名称、用途、参数和实际执行函数。
- 解析模型的工具请求，执行工具，并把结果交回模型。
- 给循环增加最大步数、工具次数、超时和错误处理。
- 区分“模型停止回答”和“任务完成”。

这一页用“12×34 等于多少”检查同一条工具链：模型请求计算，程序返回 408，再把它交回模型。算术结果容易手算，能先把消息和执行机制看清，之后再换成读取资料。下方 `ReplayModel` 的请求由我们预写，不是在测试模型会不会算术。

[返回课程入口](../README.md) · 必学主线。

先看 [minimal_loop 的实现](#impl-minimal_loop)，再观察它后面的 `messages`。工具说明给模型看，`tool.function(**args)` 才真正执行，追加 `role="tool"` 后结果才进入下一次输入。


## 1. 普通回答、JSON 和工具调用有什么区别

普通对话返回文本，例如“Agent 可以根据反馈选择下一步”。程序可以展示这段话，但不容易稳定提取其中的字段。

结构化输出让模型按约定返回数据，例如：

```json
{"answer": "Agent 根据反馈决定下一步", "needs_source": true}
```

程序可以解析 `answer` 和 `needs_source`，但还要检查字段和类型。JSON 能解析，不代表答案正确，也不代表读取过资料。

工具调用则是模型向程序提出操作请求。下面是本课程接口使用的教学示例：

```json
{
  "message": {
    "role": "assistant",
    "content": "",
    "tool_calls": [
      {"function": {"name": "read_file", "arguments": {"path": "agent.md"}}}
    ]
  }
}
```

这里的 `name` 指定工具，`arguments` 提供参数。此时模型只是提出读取请求，程序还没有打开文件。


## 2. 怎样定义一个工具

一个工具需要两部分：**给模型看的接口说明，以及程序实际调用的函数。**

以 `read_file` 为例：

| 内容 | 应该写清楚什么 |
| --- | --- |
| 名称 | `read_file` |
| 用途 | 读取允许目录中的文档正文 |
| 参数 | `path`，字符串，表示相对文档目录的路径 |
| 返回值 | 读取成功或失败、来源编号、正文 |
| 限制 | 检查路径、文件类型和大小，禁止读取目录外文件 |

工具的参数规则通常称为 schema。例如：

```json
{
  "type": "object",
  "properties": {"path": {"type": "string"}},
  "required": ["path"],
  "additionalProperties": false
}
```

它要求必须有字符串参数 `path`，并禁止额外参数。程序仍需实际执行检查；把规则写进工具说明，并不会自动限制文件访问。

用途也要写得具体：`search_documents` 找候选文档，`read_file` 读取正文。搜索返回标题，不等于正文已经读过。


## 3. 怎样执行工具并把结果交回模型

一次完整交换有五步：

1. 把用户问题、消息历史和工具说明发给模型。
2. 保存模型返回的 assistant 消息，检查是否有 `tool_calls`。
3. 取出工具名称和参数，检查工具是否存在、参数是否合法、是否允许执行。
4. 调用真实函数，把成功结果或错误保存为工具消息。
5. 把工具消息加入 `messages`，再次调用模型。

主程序的结构如下，属于教学示意：

```python
reply = model(messages, tools)
messages.append(reply)
for request in reply.get("tool_calls", []):
    result = execute(request)
    messages.append(make_tool_message(request, result))
```

`execute` 必须处理检查和异常，`make_tool_message` 必须使用模型接口要求的消息格式。不同接口的工具关联字段可能不同。

**只把工具结果打印出来不够。** 如果没有加入下一轮请求，模型不会看到它。本项目研究助手通常返回 `ok/result`，本页早期算术循环使用 `ok/data`；读代码时按具体函数的格式解析。


## 4. 工具失败后怎么办

程序应把失败也交回模型，而不是让整个循环直接崩溃。

| 工具返回 | 模型下一步可以做什么 |
| --- | --- |
| 正文读取成功 | 根据资料回答，或继续补充缺失信息 |
| 文件不存在 | 查找其他文件，或说明无法读取 |
| 搜索结果为空 | 换查询词、换来源，或说明证据不足 |
| 参数不合法 | 根据错误修正参数 |
| 重复调用 | 使用已有结果，或选择其他动作 |

这些是期望行为，需要实际运行才能知道模型是否做到。本页 `ReplayModel` 按预先给定的顺序返回消息，用来检查程序能否接收、执行和回传；它没有自主选择工具。


## 5. 给循环设置限制

| 限制 | 作用 |
| --- | --- |
| `max_steps` | 限制模型调用轮数 |
| `max_tools` | 限制工具请求次数；一轮可能有多次请求 |
| `deadline` | 在调用前后检查总运行时间 |
| 单个请求的 `timeout` | 限制某次网络请求的等待时间 |

达到限制时，记录停止原因并说明未完成的部分。调用之间检查时间，不能强制中断已经阻塞的任意函数，因此工具自身也需要超时设计。

模型没有返回工具请求、只返回文本时，可以结束循环。但还需要检查：问题是否回答、资料是否读过、引用是否支持结论。


## 6. 实际运行：为什么还需要检查完成条件

2026-10-20，本项目使用 Qwen3-0.6B 在 CPU 上回答“哪个阶段要求至少 20 道评测题？”。正确答案是 Stage 7。

| 运行方式 | 实际发生的过程 | 结果 |
| --- | --- | --- |
| 要求读资料再回答 | 一次模型调用，没有工具执行 | 回答 Stage 3，并生成未经读取支持的 `[L1]` |
| 明确要求读取 `course.md` | 一次模型调用，仍没有工具执行 | 只说来源编号是 `[L1]`，没有回答阶段 |
| 程序要求至少成功读取一次 | 三次模型调用，一次工具执行 | 读到资料后答对 Stage 7，但遗漏引用编号 |

第三次运行中，程序先拒绝没有读取记录的回答，并明确提示调用 `read_file` 读取 `course.md`。模型随后请求读取，工具返回包含 Stage 7 要求的正文，模型再回答。因此，这次成功读取包含了程序的明确引导，不能视为模型自主找到了资料。

三次运行的提示和执行规则不同，这里用于解释流程差异，不能合并为同一条件下的成功率。读取成功、事实答对和引用完整也要分开检查。


## 7. 练习与自查

试着画出：用户问题 → 模型请求 → 参数检查 → 工具执行 → 结果回传 → 下一轮回答。

然后回答：

1. 工具请求出现时，文件已经读过了吗？
2. 工具结果为什么必须进入 `messages`？
3. 一轮有两个工具请求时，步数和工具次数分别增加多少？
4. 文件不存在时，怎样让模型知道？
5. 模型停止后，还要检查什么？

先能解释本页 `minimal_loop` 的输入和每条追加消息，再考虑把算术工具换成读取资料。首课保留的真实实现使用 `read_file` 和 `search_documents` 返回预先加载的课程资料。

主线下一步：[运行程序怎样管理失败与停止](../03-agent-harness/01-harness.ipynb)。接口细节可选读：[调用真实模型](02-live-model.ipynb)。


## 代码说明

下方依次定义参数规则、参数检查、工具对象和 `minimal_loop`。`calculator` 用来提供容易核对的工具结果；`ReplayModel` 用来检查循环。`normalize_tool_message` 处理部分文本形式的工具请求，`OllamaModel` 负责模型接口适配。

按顺序运行会执行本地算术与消息格式检查。真实研究助手的运行过程已在第 6 节说明，本页回放测试不会重新产生那几次模型结果。


In [ ]:
from miniagent_course import ROOT, OUTPUT, load_lesson


In [ ]:
from __future__ import annotations
import copy, json, math, re, time, tempfile, os, sys
from dataclasses import dataclass
from pathlib import Path
from urllib.request import Request, build_opener, ProxyHandler
os.environ["PYTHONIOENCODING"] = "utf-8"

<a id="impl-schema"></a>

### 实现：schema


In [ ]:
def schema(name, description, fields):
    return {"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object", "properties": fields,
                       "required": list(fields), "additionalProperties": False}}}

<a id="impl-validate"></a>

### 实现：validate


In [ ]:
def validate(arguments, spec):
    """Deliberately small JSON Schema subset; not a general validator."""
    fields = spec["function"]["parameters"]["properties"]
    if not isinstance(arguments, dict) or set(arguments) != set(fields):
        raise ValueError("argument keys do not match schema")
    for key, rule in fields.items():
        value = arguments[key]
        if rule["type"] == "string" and (not isinstance(value, str) or not 0 < len(value) <= 500):
            raise ValueError(key + " must be a nonempty string, <=500 characters")
        if rule["type"] == "number" and (type(value) not in (int, float) or not math.isfinite(value) or abs(value) > 1e9):
            raise ValueError(key + " must be a finite number, abs <=1e9")
        if "enum" in rule and value not in rule["enum"]:
            raise ValueError(key + " is outside enum")

<a id="impl-Tool"></a>

### 实现：Tool


In [ ]:
@dataclass
class Tool:
    spec: dict
    function: object
    needs_approval: bool = False
    repeatable: bool = False

<a id="impl-calculator"></a>

### 实现：calculator


In [ ]:
def calculator(a, b, operation):
    if operation == "add":
        return a + b
    if operation == "multiply":
        return a * b
    raise ValueError("unknown operation")

<a id="impl-calculator_tool"></a>

### 实现：calculator_tool


In [ ]:
def calculator_tool():
    return Tool(schema("calculator", "两数相加或相乘", {
        "a": {"type": "number"}, "b": {"type": "number"},
        "operation": {"type": "string", "enum": ["add", "multiply"]}}), calculator)

In [ ]:
tool = calculator_tool()
print(tool.function(a=12, b=34, operation="multiply"))
print(json.dumps(tool.spec, ensure_ascii=False, indent=2))
try:
    validate({"a": True, "b": 2, "operation": "add"}, tool.spec)
except ValueError as error:
    print("bool 不是这里允许的数字：", error)

<a id="impl-call"></a>

### 实现：call


In [ ]:
def call(name, **arguments):
    return {"role": "assistant", "content": "", "tool_calls": [
        {"function": {"name": name, "arguments": arguments}}]}

<a id="impl-final"></a>

### 实现：final


In [ ]:
def final(text):
    return {"role": "assistant", "content": text}

<a id="impl-ReplayModel"></a>

### 实现：ReplayModel


In [ ]:
class ReplayModel:
    """Replay authored outputs so learners can inspect control flow offline."""
    def __init__(self, outputs):
        self.outputs = iter(copy.deepcopy(outputs))
        self.requests = []
        self.tokens = 0

    def __call__(self, messages, tools, timeout):
        self.requests.append(copy.deepcopy(messages))
        return next(self.outputs)

<a id="impl-minimal_loop"></a>

### 实现：minimal_loop


In [ ]:
def minimal_loop(question, model, max_steps=4, max_tools=4, timeout=5):
    messages = [{"role": "user", "content": question}]
    tools = {"calculator": calculator_tool()}
    deadline, used = time.monotonic() + timeout, 0
    for step in range(max_steps):
        remaining = deadline - time.monotonic()
        if remaining <= 0:
            return "timeout", messages
        try:
            reply = model(messages, [t.spec for t in tools.values()], remaining)
        except Exception as error:
            return "model_error:" + type(error).__name__, messages
        messages.append(reply)
        if time.monotonic() >= deadline:
            return "timeout", messages
        if not reply.get("tool_calls"):
            return reply.get("content", ""), messages
        for requested in reply["tool_calls"]:
            if used >= max_tools:
                return "tool_limit", messages
            used += 1
            name = "invalid"
            try:
                name = requested["function"]["name"]
                args = requested["function"]["arguments"]
                validate(args, tools[name].spec)
                result = {"ok": True, "data": tools[name].function(**args)}
            except Exception as error:
                result = {"ok": False, "error": type(error).__name__}
            messages.append({"role": "tool", "tool_name": name,
                             "content": json.dumps(result)})
    return "step_limit", messages

In [ ]:
answer, messages = minimal_loop("12×34", ReplayModel([
    call("calculator", a=12, b=34, operation="multiply"), final("408")]))
print(answer)
for message in messages:
    print(message)
assert json.loads(messages[2]["content"])["data"] == 408

<a id="change-one-rule"></a>

### 改一条规则，先预测再验证

前一个例子一次计算就够。下面故意先请求一个不存在的工具，再请求正确计算，最后返回答案；这三个决定由 `ReplayModel` 预写。

先不运行，预测：`max_tools=2` 能走到最终答案吗？改成 `max_tools=1` 呢？失败请求也占用次数，因为程序已经接收并处理了一次请求。

运行后检查 `history[2]` 中的错误、后续工具结果及最后打印的停止原因。两次工具额度时应得到 `5`；一次额度时应得到 `tool_limit`，不能再执行第二个请求。只改预算，不改预写动作，这样能看出程序规则改变了哪里。


In [ ]:
outputs = [call("unknown", a=2), call("calculator", a=2, b=3, operation="add"), final("5")]
answer, history = minimal_loop("2+3", ReplayModel(outputs))
assert answer == "5" and json.loads(history[2]["content"])["ok"] is False
print(history)
print(minimal_loop("2+3", ReplayModel(outputs), max_tools=1)[0])

<a id="impl-normalize_tool_message"></a>

### 实现：normalize_tool_message


In [ ]:
def normalize_tool_message(message, tools):
    """Some small models emit an explicit JSON action in content; parse that action only."""
    message = copy.deepcopy(message)
    if not message.get("tool_calls") and tools:
        try:
            action = json.loads(message.get("content", ""))
            names = {t["function"]["name"] for t in tools}
            if (isinstance(action, dict) and set(action) == {"name", "arguments"} and
                    action["name"] in names and isinstance(action["arguments"], dict)):
                message["tool_calls"] = [{"function": action}]
                message["content"] = ""
        except (ValueError, TypeError):
            pass
    return message

<a id="impl-OllamaModel"></a>

### 实现：OllamaModel


In [ ]:
class OllamaModel:
    """Real /api/chat adapter, fixed local endpoint, bounded HTTP timeout."""
    def __init__(self, model="qwen3:1.7b", *, max_output_tokens=512, think=False):
        self.model = model
        self.tokens = 0
        self.max_output_tokens = max_output_tokens
        self.think = think
        self.requests = []
        self.input_tokens = self.output_tokens = 0

    def __call__(self, messages, tools, timeout):
        payload = {"model": self.model, "messages": messages,
                   "tools": tools, "stream": False, "think": self.think,
                   "options": {"temperature": 0, "num_predict": self.max_output_tokens,
                               "num_ctx": 8192}}
        body = json.dumps(payload).encode()
        request = Request("http://127.0.0.1:11434/api/chat", body,
                          {"Content-Type": "application/json"})
        with build_opener(ProxyHandler({})).open(request, timeout=timeout) as response:
            result = json.load(response)
        incoming, outgoing = result.get("prompt_eval_count", 0), result.get("eval_count", 0)
        self.input_tokens += incoming
        self.output_tokens += outgoing
        self.tokens += incoming + outgoing
        self.requests.append({"request": copy.deepcopy(payload), "response": result})
        normalized = normalize_tool_message(result["message"], tools)
        self.requests[-1]["message_normalized"] = normalized != result["message"]
        return normalized